# Data cleaning and pre-processing script

## Importing dependencies and loading existing raw data

In [ ]:
# importing dependencies
from dependencies import *

In [ ]:
# loading existing raw data
df = pd.read_csv(raw_csv_filepath, encoding='utf-8-sig')
print(f"Read {len(df)} rows from {raw_csv_filepath}")

## Data cleaning

In [ ]:
# removing rows from before June 2024 (earliest observed instance of smart brick / smart play related content)
df['created_utc'] = pd.to_datetime(df['created_utc'])
df_filtered_date = df[df['created_utc'] >= '2024-06-05']
print(f"Removed {len(df)-len(df_filtered_date)} rows from before June 5th 2024. Current rows: {len(df_filtered_date)}")

In [ ]:
# removing rows from bots, moderators, and deleted accounts
excluded_authors = {
    'BrickTap',
    'LEGOIdBot', 
    'LEGOLinkBot',
    'haikusbot',
    'twitterinfo_bot',
    '[deleted]',
    'AutoModerator',
    'Clonecommando99',
    'LegoCityMaster',
    'Legoleddit',
    'CarterBricks04'}

df_filtered_author = df_filtered_date[~df_filtered_date['author'].isin(excluded_authors)]
print(f"Removed {len(df_filtered_date)-len(df_filtered_author)} rows by excluded authors. Current rows: {len(df_filtered_author)}")

In [ ]:
# removing rows where author may have mass-redacted their content
df_filtered_redacted = df_filtered_author[~df_filtered_author['text'].str.contains(
    r'This post was mass deleted and anonymized with \[Redact\]\(https://redact\.dev/home\)',
    regex=True,
    na=False)]
print(f"Removed {len(df_filtered_author)-len(df_filtered_redacted)} redacted rows. Current rows: {len(df_filtered_redacted)}")

## Preprocessing text columns for later keyword expansion and analysis

### Defining preprocessing functions

In [ ]:
# function for cleaning text for BERTopic
def preprocess_text_bert_pyabsa(text: str) -> str:
    if pd.isna(text) or text.strip() == '':
        return ''
    text = str(text)
    # remove Reddit GIF markdown e.g. ![gif](url)
    text = re.sub(r'!\[gif\]\([^)]+\)', '', text)
    # remove Markdown hyperlinks e.g. [text](url)
    text = re.sub(r'\[.*?\]\(https?://\S+\)', '', text)
    # remove bare URLs
    text = re.sub(r'https?://\S+', '', text)
    # remove Reddit quote markers (lines starting with >)
    text = re.sub(r'^>+\s*', '', text, flags=re.MULTILINE)
    # remove emojis
    text = emoji.replace_emoji(text, replace='')
    # remove markdown bold formatting while keeping the text
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    # remove markdown italic formatting while keeping the text
    text = re.sub(r'\*([^*]+)\*', r'\1', text)
    # remove markdown strikethrough formatting while keeping the text
    text = re.sub(r'~~([^~]+)~~', r'\1', text)
    # remove markdown backslash escapes e.g. \_ \* \[ 
    text = re.sub(r'\\([^\w\s])', r'\1', text)
    # normalize unicode quotes to standard ASCII equivalents
    text = text.replace('\u201c', '"').replace('\u201d', '"').replace('\u2018', "'").replace('\u2019', "'")
    # converting currency symbols to their corresponding text
    text = re.sub(r'\$+', ' dollar ', text)
    text = re.sub(r'£+', ' pound ', text)
    text = re.sub(r'€+', ' euro ', text)
    # normalize multiple whitespace characters to a single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# function for cleaning text for CorEx
def preprocess_text_corex(text: str) -> str:
    if pd.isna(text) or text.strip() == '':
        return ''
    text = str(text)
    # splitting compound words
    text = text.replace('/', ' ').replace('-', ' ')
    # remove punctuation
    text = re.sub(r"[^\w\s']", "", text)
    # lowercase all text
    text = text.lower()
    # normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # lemmatize tokens
    doc = nlp(text)
    text = ' '.join([token.lemma_ for token in doc if not token.is_space])
    return text

In [ ]:
# defining function for assigning time period for visibility, based on date
def assign_period(date):
    if date <= pd.Timestamp('2026-02-28'):
        return 't₁ | June 5, 2024 - February 28, 2026 (from first leak/reveal until release)'
    else:
        return 't₂ | March 1, 2026 - May 5, 2026 (from release until two months after release)'

### Applying pre-processing functions

In [ ]:
print("Applying preprocessing functions...")
input_column = df_filtered_redacted["text"]

In [ ]:
# applying lighter BERT preprocessing first
df_filtered_redacted["preprocessed_text_bert_pyabsa"] = input_column.apply(preprocess_text_bert_pyabsa)

# creating lemmatized version of the column to be used with pyabsa
df_filtered_redacted["preprocessed_text_bert_pyabsa_lemmatized"] = [
    ''.join(token.lemma_ + token.whitespace_ for token in doc if not token.is_space) if text.strip() else text
    for text, doc in zip(
        df_filtered_redacted["preprocessed_text_bert_pyabsa"].fillna(''),
        nlp.pipe(df_filtered_redacted["preprocessed_text_bert_pyabsa"].fillna(''), batch_size=256))]

In [ ]:
# applying stricter CorEx preprocessing on the preprocssed BERT column
df_filtered_redacted["preprocessed_text_corex"] = df_filtered_redacted["preprocessed_text_bert_pyabsa"].apply(preprocess_text_corex)

In [ ]:
# removing rows where processed text OR raw text is empty or <= 1 word
df_final = df_filtered_redacted[
    df_filtered_redacted["text"].notna() &
    df_filtered_redacted["text"].str.strip().ne('') &
    df_filtered_redacted["text"].str.strip().str.split().str.len().gt(1) &

    df_filtered_redacted["preprocessed_text_bert_pyabsa"].notna() &
    df_filtered_redacted["preprocessed_text_bert_pyabsa"].str.strip().str.split().str.len().gt(1) &

    df_filtered_redacted["preprocessed_text_corex"].notna() &
    df_filtered_redacted["preprocessed_text_corex"].str.strip().str.split().str.len().gt(1)].copy()

print(
    f"Removed {len(df_filtered_redacted) - len(df_final)} rows that had empty or 1-word text after preprocessing")

In [ ]:
# assigning time period
df_final['time_period'] = pd.to_datetime(df_filtered_redacted['created_utc']).apply(assign_period)

## Saving cleaned and preprocessed data

In [ ]:
print(f"Removed total of {len(df)-len(df_final)} rows during data cleaning")
print(f"{len(df[df['type'] == 'post'])-len(df_final[df_final['type'] == 'post'])} posts and {len(df[df['type'] == 'comment'])-len(df_final[df_final['type'] == 'comment'])} comments were removed")
print(f"Final corpus size: {len(df_final)}")
print(f"Consisting of {len(df_final[df_final['type'] == 'post'])} posts and {len(df_final[df_final['type'] == 'comment'])} comments")

In [ ]:
# saving to .csv
df_final.to_csv(
    preprocessed_csv_filepath,
    index=False,
    encoding='utf-8-sig',
    quoting=csv.QUOTE_ALL,
    escapechar='\\',
    lineterminator='\n')

# savving to .tsv
df_final.to_csv(
    preprocessed_tsv_filepath,
    index=False,
    sep='\t',
    encoding='utf-8-sig',
    quoting=csv.QUOTE_ALL,
    escapechar='\\',
    lineterminator='\n')

print(f"Succesfully saved {len(df_final)} rows to {preprocessed_csv_filepath} and {preprocessed_tsv_filepath}")